# Rule: **build_industry_sector_ratios**


**Description**

Ideal future industries with modern technologies are considered for every sector. Sector ratios are built from scratch using European data and are independent from the horizon year. These values can be thought off as the "ideal scenario" energy intensities for every carrier and sector.

The following carriers are considered:
- elec
- coal
- coke
- biomass
- methane
- hydrogen
- heat
- naphtha
- process emission
- process emission from feedstock
- (ammonia) -> if the configuration parameter industry.ammonia is set to "true", the ammonia demand is not converted to hydrogen and electricity but is considered as a separate carrier.


The additional configuration parameters associated with this rule are defined under the **industry** section of the config file and are currently expressed as EU27 aggregate values, meaning that the same energy intensity is applied across all countries.

- industry.reference_year
- industry.HVC_production_today
- industry.petrochemical_process_emissions
- industry.NH3_process_emissions
- industry.chlorine_production_today
- industry.methanol_production_today

Additional configuration related to ammonia, energy intensities and technological transformations is taken into account:
- sector.ammonia
- industry.MWh_CH4_per_tNH3_SMR
- industry.MWh_elec_per_tNH3_SMR
- industry.MWh_H2_per_tNH3_electrolysis
- industry.MWh_elec_per_tNH3_electrolysis
- industry.MWh_NH3_per_tNH3
- industry.MWh_H2_per_tCl
- industry.MWh_elec_per_tCl
- industry.MWh_CH4_per_tMeOH
- industry.MWh_elec_per_tMeOH
- industry.MWh_MeOH_per_tMeOH
- industry.MWh_elec_per_tHVC_mechanical_recycling
- industry.MWh_elec_per_tHVC_chemical_recycling
- industry.H2_DRI
- industry.elec_DRI

**Inputs**

- data/jrc_idees/archive/{database date}/EU27/`JRC-IDEES-{year}_Industry_EU27.xlsx` 
- resources/{prefix}/{name}/`ammonia_production.csv`

Note: "archive" directories and database years may vary with different versions of PyPSA-EUR/PyPSA-Spain.

**Outputs**

- resources/{prefix}/{name}/`industry_sector_ratios.csv`

These results are EU28 average specific energy consumption by carrier and industries. 

In [ ]:
######################################## Parameters

### Run
prefix = ''
name = ''

In [ ]:
##### Imports
import pandas as pd
import os 
import sys
import numpy as np
import matplotlib.pyplot as plt

##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp

##### Read params.yaml
params = xp.read_params('../params.yaml')

##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

##### Set options
pd.set_option("display.max_columns", None)

## `industry_sector_ratios.csv`  
Load the file and preview its content.

In [ ]:
file = f"industry_sector_ratios.csv"

sector_ratios = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

sector_ratios = sector_ratios.set_index(sector_ratios.columns[0])
sector_ratios

Parameters and layout colors for the graphs

In [ ]:
#################### Parameters

### Industrial sectors to plot. Choose any subset, in desired order.
sectors_to_plot = [
    "Electric arc",
    "DRI + Electric arc",
    "Integrated steelworks",
    "HVC",
    "HVC (mechanical recycling)",
    "HVC (chemical recycling)",
    "Ammonia",
    "Chlorine",
    "Methanol",
    "Other chemicals",
    "Pharmaceutical products etc.",
    "Cement",
    "Ceramics & other NMM",
    "Glass production",
    "Pulp production",
    "Paper production",
    "Printing and media reproduction",
    "Food, beverages and tobacco",
    "Alumina production",
    "Aluminium - primary production",
    "Aluminium - secondary production",
    "Other non-ferrous metals",
    "Transport equipment",
    "Machinery equipment",
    "Textiles and leather",
    "Wood and wood products",
    "Other industrial sectors",
]

energy_carriers = [
    "elec",
    "coal",
    "coke",
    "biomass",
    "methane",
    "hydrogen",
    "heat",
    "naphtha",
    "ammonia",
    "methanol"
]

############ Energy carrier colors
colors = {
    "elec": "#1f77b4",
    "coal": "#444444",
    "coke": "#8c564b",
    "biomass": "#2ca02c",
    "methane": "#ff7f0e",
    "hydrogen": "#17becf",
    "heat": "#d62728",
    "naphtha": "#9467bd",
    "ammonia": "#bcbd22",
    "methanol": "#e377c2"
}

# exclude emissions
exclude_rows = [
    "process emission",
    "process emission from feedstock",
]

How do energy intensities vary across different industrial sectors?

In [ ]:
#################### Prepare data

sector_ratios_energy = sector_ratios.drop(
    index=[r for r in exclude_rows if r in sector_ratios.index]
)

data = []

for sector in sectors_to_plot:
    for carrier in energy_carriers:
        value = sector_ratios_energy.loc[carrier, sector]
        data.append({
            "sector": sector,
            "carrier": carrier,
            "value": value
        })

df_long = pd.DataFrame(data)
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce").fillna(0)

#################### Reorder data for plotting

df_plot = df_long.pivot(index="sector", columns="carrier", values="value").fillna(0)
df_plot = df_plot.reindex(sectors_to_plot)

carriers_in_plot = [carrier for carrier in energy_carriers if carrier in df_plot.columns]
df_plot = df_plot[carriers_in_plot]

#################### Plot

fig, ax = plt.subplots(figsize=(11, 6))

y = np.arange(len(df_plot.index))
pos_cum = np.zeros(len(df_plot.index))
neg_cum = np.zeros(len(df_plot.index))

for carrier in carriers_in_plot:
    values = df_plot[carrier].values.astype(float)

    left = np.where(values >= 0, pos_cum, neg_cum)

    ax.barh(
        y,
        values,
        left=left,
        color=colors.get(carrier, "#cccccc"),
        edgecolor="none",
        label=carrier
    )

    pos_cum += np.where(values > 0, values, 0)
    neg_cum += np.where(values < 0, values, 0)

#################### Layout

ax.axvline(0, color="black", linewidth=0.8)

ax.set_title("Ideal sector ratios")
ax.set_xlabel("Energy ratios (MWh/tMaterial)")
ax.set_ylabel("Industrial sector")

ax.set_yticks(y)
ax.set_yticklabels(df_plot.index)
ax.invert_yaxis()

xmin = neg_cum.min()
xmax = pos_cum.max()
xrange = xmax - xmin

if np.isclose(xrange, 0):
    xrange = 1

pad = 0.05 * xrange
ax.set_xlim(xmin - pad, xmax + pad)

ax.legend(
    title="Energy carrier",
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
    frameon=False
)

ax.set_facecolor("white")
fig.patch.set_facecolor("white")

plt.tight_layout()
plt.show()